# P03 — Воспроизводимый эксперимент / Reproducible experiment

CPU, NumPy + scikit-learn. Полностью выполненный пример. / Fully worked example.

Цель: сравнить baseline и модель на одинаковых разбиениях, исключив пересечение групп. / Compare a baseline and a model on identical splits without group overlap.

In [ ]:
import json, hashlib, platform
import numpy as np
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

## Данные / Data
Практика использует GroupShuffleSplit и StandardScaler; в HW03 реализуйте заданный алгоритм NumPy самостоятельно. Разбиения практики и ДЗ могут отличаться. / Practice uses sklearn; implement the specified NumPy algorithm in HW03. Their split memberships may differ.

60 условных объектов, по 4 связанных измерения. Признаки доступны в момент прогноза; группа служит только для разбиения. Данные искусственные, качество не переносится на реальную задачу. / 60 fictional entities with 4 related measurements each. Features are available at prediction time; the group is used only for splitting. These synthetic results do not establish real-world performance.

In [ ]:
# Учебные синтетические данные / Synthetic teaching data
rng = np.random.default_rng(2026)
groups = np.repeat(np.arange(60), 4)
latent = rng.normal(size=(60, 3))
X = latent[groups] + rng.normal(0, .12, size=(240, 3))
X *= np.array([1., 100., .01])
y = (latent[groups, 0] + .6 * latent[groups, 1] > 0).astype(int)
row_ids = np.arange(len(y))
assert X.shape == (240, 3) and set(np.unique(y)) == {0, 1}

## Разбиение по группам / Group split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def split_groups(groups, seed=42):
    rows = np.arange(len(groups))
    first = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=seed)
    rest, test = next(first.split(rows, groups=groups))
    second = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=seed)
    train_rel, val_rel = next(second.split(rest, groups=groups[rest]))
    return rest[train_rel], rest[val_rel], test

## Статистики только Train / Train-only statistics

In [ ]:
scaler_demo = StandardScaler().fit([[0.], [2.]])
print(scaler_demo.mean_, scaler_demo.scale_, scaler_demo.transform([[100.]]))
assert np.allclose(scaler_demo.transform([[100.]]), [[99.]])

## Pipeline

In [ ]:
def make_model(C):
    return Pipeline([('scale', StandardScaler()),
                     ('model', LogisticRegression(C=C, solver='lbfgs', max_iter=1000, random_state=42))])

## Выбор по Validation / Validation selection

In [ ]:
def choose_candidate(records):
    return min(records, key=lambda row: (-row['val_f1'], row['C']))['C']

## Журнал / Experiment log

In [ ]:
def make_log(train_ids, val_ids, test_ids, seed, selected_C, val_f1, test_f1):
    return {'assignment':'HW03_Experiment', 'seed':int(seed),
            'train_ids':list(map(int, train_ids)), 'val_ids':list(map(int, val_ids)),
            'test_ids':list(map(int, test_ids)), 'selected_C':float(selected_C),
            'val_f1':float(val_f1), 'test_f1':float(test_f1),
            'numpy_version':np.__version__, 'sklearn_version':sklearn.__version__}

## Проверка нормализации / Check standardization
Train = [0,2], Validation = [100]. Каким станет 100? / Train = [0,2], Validation = [100]. What does 100 become?

In [ ]:
assert np.allclose(scaler_demo.transform([[100.]]), [[99.]])

## Эксперимент / Experiment
Основная метрика — macro-F1. Порог по умолчанию модели фиксирован. При точном равенстве метрик выбираем меньший C. / Primary metric: macro-F1. The model default threshold is fixed. Exact metric ties select the smaller C.

In [ ]:
train, val, test = split_groups(groups, seed=42)
for a, b in [(train, val), (train, test), (val, test)]:
    assert not set(groups[a]) & set(groups[b])
for part in (train, val, test):
    assert set(np.unique(y[part])) == {0, 1}, 'Need both classes for this teaching example'
baseline = DummyClassifier(strategy='most_frequent').fit(X[train], y[train])
print('Baseline validation macro-F1:', f1_score(y[val], baseline.predict(X[val]), average='macro', zero_division=0))
records, models = [], {}
for C in [.1, 1., 10.]:
    model = make_model(C).fit(X[train], y[train])
    models[C] = model
    records.append({'C':C, 'val_f1':float(f1_score(y[val], model.predict(X[val]), average='macro', zero_division=0))})
selected_C = choose_candidate(records)
print('Validation results:', records, 'Selected C:', selected_C)
# Выбор зафиксирован ДО доступа к Test / Freeze selection BEFORE using Test
selected = models[selected_C]
test_f1 = f1_score(y[test], selected.predict(X[test]), average='macro', zero_division=0)
baseline_test = f1_score(y[test], baseline.predict(X[test]), average='macro', zero_division=0)
log = make_log(row_ids[train], row_ids[val], row_ids[test], 42, selected_C,
               next(r['val_f1'] for r in records if r['C'] == selected_C), test_f1)
log['baseline_test_f1'] = float(baseline_test)
log['data_sha256'] = hashlib.sha256(X.tobytes()+y.tobytes()+groups.tobytes()).hexdigest()
print(json.dumps(log, ensure_ascii=False))

## Обсуждение / Discussion
1. Почему разбиение по строкам не проверяет перенос на новые группы? / Why does splitting rows fail to assess generalization to unseen groups?
2. Что изменится, если fit scaler выполнить на всех данных? / What changes if the scaler is fitted on all data?
3. Почему низкий Test не разрешает выбирать другой C по Test? / Why does a low test score not justify selecting another C using Test?
4. Для курсового проекта запишите единицу разбиения и доступность каждого признака. / For your course project, record the split unit and when each feature becomes available.